In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error


In [ ]:
df = pd.read_csv("../../data/FRB_H15.csv").dropna()

#rename columns
df.rename(columns={"Series Description": "Date", "Market yield on U.S. Treasury securities at 1-month  constant maturity, quoted on investment basis": "0Y1M", "Market yield on U.S. Treasury securities at 3-month  constant maturity, quoted on investment basis": "0Y3M", "Market yield on U.S. Treasury securities at 6-month  constant maturity, quoted on investment basis": "0Y6M", "Market yield on U.S. Treasury securities at 1-year  constant maturity, quoted on investment basis": "1Y", "Market yield on U.S. Treasury securities at 2-year  constant maturity, quoted on investment basis": "2Y", "Market yield on U.S. Treasury securities at 3-year  constant maturity, quoted on investment basis": "3Y", "Market yield on U.S. Treasury securities at 5-year  constant maturity, quoted on investment basis": "5Y", "Market yield on U.S. Treasury securities at 7-year  constant maturity, quoted on investment basis": "7Y", "Market yield on U.S. Treasury securities at 10-year  constant maturity, quoted on investment basis": "10Y", "Market yield on U.S. Treasury securities at 20-year constant maturity, quoted on investment basis": "20Y", "Market yield on U.S. Treasury securities at 30-year  constant maturity, quoted on investment basis": "30Y"}, inplace=True)
df.rename(columns={"Market yield on U.S. Treasury securities at 20-year  constant maturity, quoted on investment basis": "20Y"}, inplace=True)

#make index to datetime for timeseries
#df["Date"] = pd.to_datetime(df["Date"])
# dont make timeseries data for xgboost

#df.set_index("Date", inplace=True)
df = df.reset_index(drop=True)
df.head(6)

#### Data Processing

In [ ]:
maturities = []
matList = ["0Y1M", "0Y3M", "0Y6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]

for col in df.columns:
    maturities.append(df[col])

# 1D table array, the given index is the same index in matlist
tables = []

#lag features for all maturities, 5 lags for each maturity, 55 features in total
features = [
    "0Y1M_1", "0Y1M_2", "0Y1M_3", "0Y1M_4", "0Y1M_5",
    "0Y3M_1", "0Y3M_2", "0Y3M_3", "0Y3M_4", "0Y3M_5",
    "0Y6M_1", "0Y6M_2", "0Y6M_3", "0Y6M_4", "0Y6M_5",
    "1Y_1", "1Y_2", "1Y_3", "1Y_4", "1Y_5",
    "2Y_1", "2Y_2", "2Y_3", "2Y_4", "2Y_5",
    "3Y_1", "3Y_2", "3Y_3", "3Y_4", "3Y_5",
    "5Y_1", "5Y_2", "5Y_3", "5Y_4", "5Y_5",
    "7Y_1", "7Y_2", "7Y_3", "7Y_4", "7Y_5",
    "10Y_1", "10Y_2", "10Y_3", "10Y_4", "10Y_5",
    "20Y_1", "20Y_2", "20Y_3", "20Y_4", "20Y_5",
    "30Y_1", "30Y_2", "30Y_3", "30Y_4", "30Y_5",  
]
all_cols = features + ["target"]


for mat in matList:

    #arrays for data in each horizon, so rows1 has table with 
    rows1 = []
    rows2 = []
    rows3 = []



    for i in range(4, len(df.index)-1):
        target = df[mat].iloc[i+1]

        lagAll = []
        for target_mat in matList:
            lag1 = df[target_mat].iloc[i]
            lag2 = df[target_mat].iloc[i-1]
            lag3 = df[target_mat].iloc[i-2]
            lag4 = df[target_mat].iloc[i-3]
            lag5 = df[target_mat].iloc[i-4]
            lagAll.extend([lag1, lag2, lag3, lag4, lag5])
        lagAll.extend([target])

        #adds a row with 5 lags for each maturity, and then target value for the specific maturity that we are creating data for
        rows1.append(lagAll)

    for i in range(4, len(df.index)-5):
        target = df[mat].iloc[i+5]

        lagAll = []
        for target_mat in matList:
            lag1 = df[target_mat].iloc[i]
            lag2 = df[target_mat].iloc[i-1]
            lag3 = df[target_mat].iloc[i-2]
            lag4 = df[target_mat].iloc[i-3]
            lag5 = df[target_mat].iloc[i-4]
            lagAll.extend([lag1, lag2, lag3, lag4, lag5])
        lagAll.extend([target])

        #adds a row with 5 lags for each maturity, and then target value for the specific maturity that we are creating data for
        rows2.append(lagAll)
    for i in range(4, len(df.index)-20):
        target = df[mat].iloc[i+20]

        lagAll = []
        for target_mat in matList:
            lag1 = df[target_mat].iloc[i]
            lag2 = df[target_mat].iloc[i-1]
            lag3 = df[target_mat].iloc[i-2]
            lag4 = df[target_mat].iloc[i-3]
            lag5 = df[target_mat].iloc[i-4]
            lagAll.extend([lag1, lag2, lag3, lag4, lag5])
        lagAll.extend([target])

        #adds a row with 5 lags for each maturity, and then target value for the specific maturity that we are creating data for
        rows3.append(lagAll)
    
    df1 = pd.DataFrame(rows1, columns=all_cols)
    df2 = pd.DataFrame(rows2, columns=all_cols)
    df3 = pd.DataFrame(rows3, columns=all_cols)


    tables.append([df1, df2, df3])
        
#now tables[i] gives the ith maturity, and tables[i][0] gives the 1-day ahead table
# , tables[i][1] gives the 5-day ahead table, and tables[i][2] gives the 20-day ahead table
        

#### Train Models

In [ ]:
xy = []

for i in range(len(tables)):
    x1 = tables[i][0][features]
    y1 = tables[i][0]["target"]

    x2 = tables[i][1][features]
    y2 = tables[i][1]["target"]

    x3 = tables[i][2][features]
    y3 = tables[i][2]["target"]
    
    xy.append([(x1, y1), (x2, y2), (x3, y3)])

x_train = [[] for i in range(len(matList))]
y_train = [[] for i in range(len(matList))]
x_test = [[] for i in range(len(matList))]
y_test = [[] for i in range(len(matList))]

for i in range(len(matList)):
    # xy[i] is a list of 3 tuples, each tuple is (x,y) for 1,5,20 day forecast
    for j in range(3):
        x,y = xy[i][j]
        split = int(len(x)*0.8)
        x_train[i].append(x[:split])
        y_train[i].append(y[:split])
        x_test[i].append(x[split:])
        y_test[i].append(y[split:])


#models is a list of 3 lists, first index is maturity, second index is 1,5,20 day forecast
models = [[] for i in range(len(matList))]

for i in range(len(x_train)):
    for j in range(3):
        model = xgb.XGBRegressor(
            objective='reg:squarederror',
            n_estimators = 500,
            learning_rate = 0.02,
            max_depth = 3,
            subsample = 0.8,
            colsample_bytree = 0.8, 
            reg_lambda = 5, 
            random_state = 7
        )
        model.fit(x_train[i][j], y_train[i][j])
        models[i].append(model)

predictions = [[] for i in range(len(matList))]
for i in range(len(models)):
    pred = []
    for j in range(3):
        pred.append(models[i][j].predict(x_test[i][j]))
    predictions[i] = pred

rmse = [[] for i in range(len(matList))]
for i in range(len(predictions)):
    for j in range(3):
        rmse[i].append(root_mean_squared_error(y_test[i][j], predictions[i][j]))

for i in range(len(matList)):
    print(f"  Maturity: {matList[i]}")
    for j in range(3):
        if j == 0:
            print(f"  Forecast {j+1} day, RMSE: {rmse[i][j]}")
        elif j == 1:
            print(f"  Forecast {5} day, RMSE: {rmse[i][j]}")
        else:
            print(f"  Forecast {20} day, RMSE: {rmse[i][j]}")

#rolling window forecast for 1, 5, and 20 day horizons
horizons = [1, 5, 20]

#indexes for windows
windowStarts = range(len(df) - 41, len(df) - 20)

def rollingMetrics(errors, predictedChanges, actualChanges):
    rmse = 100 * np.sqrt(np.mean(np.array(errors)**2))
    mae = 100 * np.mean(np.abs(errors))
    directionalAccuracy = 100 * np.mean(np.sign(predictedChanges) == np.sign(actualChanges))
    return rmse, mae, directionalAccuracy

#store all metrics
metrics = [[[] for _ in range(3)] for _ in range(len(matList))]

#aggregate metrics to analyze error per horizon
sum = [[0.0,0.0,0.0] for _ in range(3)]

for i in range(len(matList)):
    
    errors = [[], [], []]
    predictedChanges = [[], [], []]
    actualChanges = [[], [], []]
    #first array is horizon, second array is the rolling window number

    for start in windowStarts:
        for j in range(len(horizons)):
            horizon = horizons[j]
            table = tables[i][j]
            trainEnd = start - horizon - 3
            model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=500, learning_rate=0.02, max_depth=3, subsample=0.8, colsample_bytree=0.8, reg_lambda=5, random_state=7)
            model.fit(table[features].iloc[:trainEnd], table["target"].iloc[:trainEnd])
            predicted = model.predict(table[features].iloc[[start - 4]])[0]
            actual = df[matList[i]].iloc[start + horizon]
            current = df[matList[i]].iloc[start]

            #predicted changes and actual changes are for directional accuracy
            errors[j].append(predicted - actual)
            predictedChanges[j].append(predicted - current)
            actualChanges[j].append(actual - current)

    print(f"Maturity: {matList[i]}")
    for j in range(len(horizons)):
        rmse, mae, directionalAccuracy = rollingMetrics(errors[j], predictedChanges[j], actualChanges[j])
        metrics[i][j].append([rmse, mae, directionalAccuracy])
        sum[j][0]+=rmse**2
        sum[j][1]+=mae
        sum[j][2]+=directionalAccuracy
        print("Forecast ", horizons[j], " day, RMSE: ", rmse, " MAE: ", mae, " Directional Accuracy: ", directionalAccuracy, "%")
    print("\n")

for i in range(len(sum)):
    for j in range(3):
        sum[i][j]= sum[i][j]/len(matList)
    sum[i][0] = np.sqrt(sum[i][0])
    print(f"Horizon: {horizons[i]} day, rmse: {sum[i][0]}, MAE: {sum[i][1]}, Directional Accuracy: {sum[i][2]} \n")


#### Model Tuning